# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [7]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Audit window: February 2026")

Connected to FlyRank warehouse.
Audit window: February 2026


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load February 2026 search performance data

feb_daily = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {FEB}
WHERE gsc_data_available IS TRUE
""").df()

print("Rows loaded:", len(feb_daily))
print("Unique clients:", feb_daily["client_hash_id"].nunique())
print("Unique content items:", feb_daily["content_hash_id"].nunique())

# Inspect key-field distributions

key_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

display(feb_daily[key_fields].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T)

# Measure zero values and skewness

distribution_summary = pd.DataFrame({
    "zero_count": (feb_daily[key_fields] == 0).sum(),
    "zero_percent": (feb_daily[key_fields] == 0).mean() * 100,
    "skewness": feb_daily[key_fields].skew()
})

display(distribution_summary.round(2))

Rows loaded: 2621783
Unique clients: 46
Unique content items: 153559


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
gsc_impressions,2621783.0,68.704741,219.427623,1.0,4.000000,15.000000,58.000000,163.0,289.0,806.0,74688.0
gsc_clicks,2621783.0,0.223590,1.187638,0.0,0.000000,0.000000,0.000000,1.0,1.0,4.0,175.0
gsc_avg_position,2621782.0,12.098910,16.049805,0.0,3.142857,6.666667,13.580645,30.5,46.0,80.6,633.0


,zero_count,zero_percent,skewness
gsc_impressions,0,0.00,52.93
gsc_clicks,2322893,88.60,33.10
gsc_avg_position,179953,6.86,3.18


### Distribution findings

The February 2026 dataset contains 2,621,783 daily records, covering 46 unique clients and 153,559 unique content items.

- Impressions have a mean of 68.70 and a median of 15, with a maximum of 74,688. The high skewness of 52.93 indicates a very heavy right tail.
- Clicks are zero in 88.60% of records. Their skewness of 33.10 indicates that most records have very few clicks, while a small number have much higher values.
- Average position has a mean of 12.10 and a median of 6.67. Its skewness of 3.18 indicates a right-skewed distribution.
- Average position contains one missing value.

### Conclusion

The key fields show highly uneven distributions and heavy tails. Therefore, averages alone may not represent typical records accurately. Percentiles, medians, and distribution-aware analysis should also be considered when evaluating signals.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal test approach

Three observable relationships will be tested using February 2026 search performance data.

1. **Position and CTR:** Do content items with better average search positions have higher weighted CTR?
2. **Impressions and clicks:** Do content items with higher impression volumes also receive more clicks?
3. **Impression volume and CTR:** Does higher impression volume correspond to better click-through efficiency?

Each test uses grouped comparisons, displays the sample size, and receives a verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.

The analysis is descriptive and measures associations, not causation.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Aggregate daily records to one row per client-content pair

audit_daily = feb_daily.copy()

audit_daily["valid_position"] = (
    audit_daily["gsc_avg_position"].notna()
    & (audit_daily["gsc_avg_position"] > 0)
    & (audit_daily["gsc_impressions"] > 0)
)

audit_daily["position_weighted"] = np.where(
    audit_daily["valid_position"],
    audit_daily["gsc_avg_position"] * audit_daily["gsc_impressions"],
    0
)

audit_daily["position_weight"] = np.where(
    audit_daily["valid_position"],
    audit_daily["gsc_impressions"],
    0
)

monthly = audit_daily.groupby(
    ["client_hash_id", "content_hash_id"]
).agg(
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    position_weighted=("position_weighted", "sum"),
    position_weight=("position_weight", "sum")
).reset_index()

monthly["avg_position"] = (
    monthly["position_weighted"] /
    monthly["position_weight"].replace(0, np.nan)
)

monthly["ctr"] = monthly["clicks"] / monthly["impressions"]

monthly = monthly[monthly["impressions"] > 0].copy()

print("Monthly content records:", len(monthly))


# Helper function for verdicts

def signal_verdict(values, expected):
    differences = np.diff(np.asarray(values, dtype=float))

    if np.all(np.abs(differences) < 1e-12):
        return "FALSE"

    if expected == "decreasing":
        if np.all(differences < 0):
            return "CONFIRMED"
        elif np.all(differences > 0):
            return "OPPOSITE"

    if expected == "increasing":
        if np.all(differences > 0):
            return "CONFIRMED"
        elif np.all(differences < 0):
            return "OPPOSITE"

    return "MIXED"


# TEST 1: Better search position and higher CTR

position_data = monthly.dropna(subset=["avg_position"]).copy()

position_data["position_band"] = pd.cut(
    position_data["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"]
)

test1 = position_data.groupby(
    "position_band", observed=True
).agg(
    n=("content_hash_id", "size"),
    impressions=("impressions", "sum"),
    clicks=("clicks", "sum")
).reset_index()

test1["weighted_ctr_pct"] = (
    test1["clicks"] / test1["impressions"] * 100
)

display(test1.round(3))

assert (test1["n"] >= 50).all(), "A position bucket has fewer than 50 records."

verdict1 = signal_verdict(
    test1["weighted_ctr_pct"], "decreasing"
)

print("TEST 1 VERDICT:", verdict1)


# TEST 2: Higher impressions and more clicks

monthly["impression_quartile"] = pd.qcut(
    monthly["impressions"].rank(method="first"),
    q=4,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4 - Highest"]
)

test2 = monthly.groupby(
    "impression_quartile", observed=True
).agg(
    n=("content_hash_id", "size"),
    median_impressions=("impressions", "median"),
    median_clicks=("clicks", "median")
).reset_index()

display(test2.round(2))

assert (test2["n"] >= 50).all(), "An impression bucket has fewer than 50 records."

verdict2 = signal_verdict(
    test2["median_clicks"], "increasing"
)

print("TEST 2 VERDICT:", verdict2)


# TEST 3: Higher impressions and better CTR

test3 = monthly.groupby(
    "impression_quartile", observed=True
).agg(
    n=("content_hash_id", "size"),
    impressions=("impressions", "sum"),
    clicks=("clicks", "sum")
).reset_index()

test3["weighted_ctr_pct"] = (
    test3["clicks"] / test3["impressions"] * 100
)

display(test3.round(3))

assert (test3["n"] >= 50).all(), "An impression bucket has fewer than 50 records."

verdict3 = signal_verdict(
    test3["weighted_ctr_pct"], "increasing"
)

print("TEST 3 VERDICT:", verdict3)

Monthly content records: 153559


,position_band,n,impressions,clicks,weighted_ctr_pct
0,1-3,15376,34515630,137228,0.398
1,4-10,75008,103186325,349459,0.339
2,11-20,32133,21237356,66836,0.315
3,21+,29439,21184517,32629,0.154


TEST 1 VERDICT: CONFIRMED


,impression_quartile,n,median_impressions,median_clicks
0,Q1 - Lowest,38390,3.0,0.0
1,Q2,38390,43.0,0.0
2,Q3,38389,303.0,0.0
3,Q4 - Highest,38390,2256.0,5.0


TEST 2 VERDICT: MIXED


,impression_quartile,n,impressions,clicks,weighted_ctr_pct
0,Q1 - Lowest,38390,166400,1405,0.844
1,Q2,38390,1939215,5649,0.291
2,Q3,38389,13319108,32222,0.242
3,Q4 - Highest,38390,164704199,546929,0.332


TEST 3 VERDICT: MIXED


### Signal test findings and verdicts

**Test 1: Search position and CTR — CONFIRMED**

Weighted CTR decreases across the position bands:
- Positions 1–3: 0.398%
- Positions 4–10: 0.339%
- Positions 11–20: 0.315%
- Positions 21+: 0.154%

This supports the expected relationship that better average search positions are associated with higher CTR.

**Test 2: Impressions and clicks — MIXED**

Median clicks are zero in the first three impression quartiles and increase to 5 in the highest quartile.

This suggests that higher impression volumes can be associated with more clicks, but the relationship is not consistently visible across all quartiles.

**Test 3: Impressions and CTR — MIXED**

Weighted CTR changes across impression quartiles:
- Q1: 0.844%
- Q2: 0.291%
- Q3: 0.242%
- Q4: 0.332%

CTR initially decreases and then increases in the highest impression quartile. Therefore, higher impression volume does not consistently produce better CTR.

### Overall observation

All three tests have more than 50 observations per bucket. Search position shows the clearest relationship with CTR. Impression volume has a less consistent relationship with clicks and CTR.

These findings describe associations in the February 2026 dataset and do not establish causation.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test: In striking distance

FlyRank uses an `in_striking_distance` rule to identify content that may be close to reaching the first page of search results.

**Rule assumption:** Content with an average search position between 11 and 30 should have a stronger CTR opportunity than content ranking beyond position 30.

**Test:** Compare impression-weighted CTR for content in positions 11–30 against content beyond position 30.

The test uses February 2026 aggregated data. It evaluates the position assumption only, not the complete production flag.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Test the in_striking_distance assumption

flag_data = monthly.dropna(subset=["avg_position"]).copy()

flag_data["position_group"] = np.where(
    flag_data["avg_position"].between(11, 30),
    "Striking distance (11-30)",
    np.where(
        flag_data["avg_position"] > 30,
        "Beyond position 30",
        "Positions 1-10"
    )
)

# Compare only the two relevant groups

flag_test = flag_data[
    flag_data["position_group"].isin([
        "Striking distance (11-30)",
        "Beyond position 30"
    ])
].groupby("position_group").agg(
    n=("content_hash_id", "size"),
    impressions=("impressions", "sum"),
    clicks=("clicks", "sum")
).reset_index()

flag_test["weighted_ctr_pct"] = (
    flag_test["clicks"] / flag_test["impressions"] * 100
)

display(flag_test.round(3))

# Check sample size and compare CTR

assert (flag_test["n"] >= 50).all(), "Insufficient sample size."

ctr_striking = flag_test.loc[
    flag_test["position_group"] == "Striking distance (11-30)",
    "weighted_ctr_pct"
].iloc[0]

ctr_beyond = flag_test.loc[
    flag_test["position_group"] == "Beyond position 30",
    "weighted_ctr_pct"
].iloc[0]

if ctr_striking > ctr_beyond:
    flag_verdict = "CONFIRMED"
elif ctr_striking < ctr_beyond:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "FALSE"

print("Striking-distance CTR:", round(ctr_striking, 3), "%")
print("Beyond-position-30 CTR:", round(ctr_beyond, 3), "%")
print("FLAG-LINKED VERDICT:", flag_verdict)

,position_group,n,impressions,clicks,weighted_ctr_pct
0,Beyond position 30,17224,8006432,7850,0.098
1,Striking distance (11-30),39022,31261406,80699,0.258


Striking-distance CTR: 0.258 %
Beyond-position-30 CTR: 0.098 %
FLAG-LINKED VERDICT: CONFIRMED


### Flag-linked test: Striking-distance signal

**Signal tested:** Content ranking in positions 11–30 is expected to have stronger search engagement than content ranking beyond position 30.

**Results:**
- Striking-distance content (positions 11–30): 39,022 records, 0.258% weighted CTR.
- Beyond-position-30 content: 17,224 records, 0.098% weighted CTR.

**Verdict: CONFIRMED**

The striking-distance group has approximately 2.63 times the weighted CTR of the beyond-position-30 group. This supports the flag's assumption that content ranking in positions 11–30 has stronger existing search engagement and may offer an improvement opportunity.

**Limitation:** This is an observed association, not proof that improving search position will automatically increase clicks. Other factors, such as search intent and content relevance, may also influence CTR.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The content team should prioritize reviewing pages ranking in positions 11–30 because they already receive stronger search engagement and may have opportunities to improve visibility. Pages ranking beyond position 30 may need a deeper review of content relevance, search intent, and quality before deciding on an action. These signals should guide investigation rather than be treated as guaranteed outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.